In [ ]:
!git clone https://github.com/siddharthyadaviitdelhi/multi-source-enterprise-rag.git

In [ ]:
%cd /content/multi-source-enterprise-rag
!git fetch origin main
!git reset --hard origin/main
%pip install -r requirements.txt

In [ ]:
%cd multi-source-enterprise-rag
%pip install -r requirements.txt

In [ ]:
from getpass import getpass
import os

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

In [ ]:
!find . -name "*.pyc" -delete
!find . -name "__pycache__" -delete

In [ ]:
!PYTHONPATH=. python -m pytest tests -q

In [ ]:
!PYTHONPATH=. python evaluation/evaluate.py

In [ ]:
!PYTHONPATH=. python evaluation/evaluate.py

In [ ]:
%pip install -q python-pptx

import os
import sys

# 1. Force the new, active model in the environment
os.environ["GROQ_MODEL"] = "openai/gpt-oss-20b"

# 2. Force override the config file directly in memory
try:
    import config
    config.GROQ_MODEL = "openai/gpt-oss-20b"
    config.LLM_MODEL = "openai/gpt-oss-20b"
except ImportError:
    pass

from ingestion.csv_loader import load_csv
from ingestion.common import chunk_document
from rag.pipeline import RAGPipeline

# 3. Load sample knowledge from the CSV file
docs = load_csv("data/sample/sample_policy.csv")

# 4. Split documents into smaller chunks
chunks = []
for doc in docs:
    chunks.extend(
        chunk_document(doc, chunk_size=900, overlap=150)
    )
print("Number of chunks:", len(chunks))

# 5. Create the RAG pipeline
rag = RAGPipeline(chunks)

# 6. Ask a question using hybrid retrieval
result = rag.answer(
    "What is the default retention period for support tickets?",
    mode="hybrid",
    top_k=5,
    use_reranker=False
)

# 7. Display the answer and its evidence
print("\nANSWER:")
print(result["answer"])
print("\nSOURCES:")
for source in result["sources"]:
    print(f"[{source['id']}] {source['source']} — {source['location']}")
    print(source["text"])
    print("-" * 50)

In [ ]:
!pip install -q gradio

import gradio as gr
import os
from ingestion.pdf_loader import load_pdf
from ingestion.csv_loader import load_csv
from ingestion.pptx_loader import load_pptx
from ingestion.common import chunk_document
from rag.pipeline import RAGPipeline

# Global variable to hold our pipeline in memory
global_rag = None

def process_document(file_path):
    global global_rag
    if not file_path:
        return "⚠️ Please upload a file first."

    ext = os.path.splitext(file_path)[-1].lower()

    try:
        if ext == ".pdf":
            docs = load_pdf(file_path)
        elif ext == ".csv":
            docs = load_csv(file_path)
        elif ext == ".pptx":
            docs = load_pptx(file_path)
        else:
            return f"❌ Unsupported file type: {ext}. Please upload a PDF, CSV, or PPTX."

        chunks = []
        for doc in docs:
            chunks.extend(chunk_document(doc, chunk_size=900, overlap=150))

        global_rag = RAGPipeline(chunks)
        return f"✅ Success! Knowledge base built with {len(chunks)} chunks."
    except Exception as e:
        return f"❌ Error processing file: {str(e)}"

def answer_question(query, mode, use_reranker):
    global global_rag
    if global_rag is None:
        return "⚠️ Please build the knowledge base first by uploading a document.", ""

    if not query:
        return "⚠️ Please enter a question.", ""

    try:
        result = global_rag.answer(
            query,
            mode=mode,
            top_k=5,
            use_reranker=use_reranker
        )

        ans = result["answer"]
        srcs = ""
        for s in result["sources"]:
            srcs += f"[{s['id']}] {s['source']} — {s['location']}\n{s['text']}\n" + "-"*50 + "\n"

        return ans, srcs
    except Exception as e:
        return f"❌ Error generating answer: {str(e)}", ""

# Build the Web UI
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🏢 Enterprise Multi-Source RAG")
    gr.Markdown("Upload your enterprise documents (PDF, CSV, or PPTX) and ask questions to get AI-generated, citation-backed answers.")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 1. Ingestion")
            file_in = gr.File(label="Upload Document (PDF/CSV/PPTX)", type="filepath")
            build_btn = gr.Button("Build Knowledge Base", variant="primary")
            status_out = gr.Textbox(label="System Status", interactive=False)

        with gr.Column(scale=2):
            gr.Markdown("### 2. Retrieval & QnA")
            query_in = gr.Textbox(label="Ask a question about the document", placeholder="e.g., What is the retention policy?")

            with gr.Row():
                mode_in = gr.Dropdown(choices=["hybrid", "semantic", "sentence_window"], value="hybrid", label="Retrieval Mode")
                rerank_in = gr.Checkbox(label="Enable Re-ranker", value=False)

            ask_btn = gr.Button("Submit Query", variant="primary")

    with gr.Row():
        ans_out = gr.Textbox(label="AI Answer", lines=5, interactive=False)
        src_out = gr.Textbox(label="Source Evidence", lines=10, interactive=False)

    # Wire up the buttons
    build_btn.click(fn=process_document, inputs=[file_in], outputs=[status_out])
    ask_btn.click(fn=answer_question, inputs=[query_in, mode_in, rerank_in], outputs=[ans_out, src_out])

# The share=True flag magically generates a public, clickable Gradio link!
demo.launch(share=True)